#DS

##Clases

###DummyModel

In [0]:
"""
Módulo para modelo dummy (simulación de modelo de ML mientras no se tiene el real).
"""
import numpy as np

class DummyModel:
    """
    Modelo dummy que devuelve probabilidad fija para pruebas.
    Serializable con pickle y compatible con MLflow (como sklearn).
    """
    def __init__(self, prob_class1=0.15):
        self.prob_class1 = prob_class1
        # Simula el atributo que algunos modelos reales tienen
        self.feature_names_in_ = None  # Se puede asignar si se conoce

    def predict_proba(self, X):
        """
        Devuelve matriz de probabilidades [P(clase0), P(clase1)].
        X puede ser numpy array o pandas DataFrame (se ignora).
        """
        n = len(X)
        prob_ones = np.full(n, self.prob_class1)
        prob_zeros = 1 - prob_ones
        return np.column_stack((prob_zeros, prob_ones))

    def predict(self, X):
        """Devuelve clase predicha (1 si probabilidad > 0.5)."""
        return (self.predict_proba(X)[:, 1] >= 0.5).astype(int)


def create_dummy_model(prob_class1=0.15):
    """Factoría para crear una instancia del modelo dummy."""
    return DummyModel(prob_class1=prob_class1)

##Load_Model

Propósito: Cargar y validar el modelo entrenado (pickle) desde el storage.

In [0]:
# Databricks notebook source
# ============================================================================
# 01_load_model.ipynb
# Propósito: Cargar modelo CatBoost desde pickle y registrarlo en Unity Catalog
#            con la firma (signature) requerida por UC. Incluye fallback a
#            FEATURE_NAMES desde config si el modelo no expone feature_names.
# ============================================================================

# COMMAND ----------

# Instalación de dependencias (igual que en Notebook Sherly)
%pip install catboost optuna xgboost lightgbm category_encoders openpyxl fsspec

# COMMAND ----------

%run ../config/config

# COMMAND ----------

import sys
import time
import pickle
import os
import numpy as np
import pandas as pd
import mlflow
import mlflow.catboost
import mlflow.tracking._model_registry.utils
mlflow.tracking._model_registry.utils._get_registry_uri_from_spark_session = lambda: "databricks-uc"

from mlflow.tracking import MlflowClient
from mlflow.models import infer_signature
from mlflow.exceptions import MlflowException
from pyspark.sql import SparkSession

sys.path.insert(0, str(PROJECT_PATH))
from utils.logger import MLOpsLogger

# ==================================================
# Definición del modelo dummy (clase simple y detectable)
# ==================================================
class DummyModel:
    """
    Modelo dummy que devuelve probabilidad fija para pruebas.
    Serializable con pickle y compatible con MLflow (como sklearn).
    """

    def __init__(self, prob_class1=0.15):
        self.prob_class1 = prob_class1
        # Simula el atributo que algunos modelos reales tienen
        self.feature_names_in_ = None  # Se puede asignar si se conoce

    def predict_proba(self, X):
        """
        Devuelve matriz de probabilidades [P(clase0), P(clase1)].
        X puede ser numpy array o pandas DataFrame (se ignora).
        """
        n = len(X)
        prob_ones = np.full(n, self.prob_class1)
        prob_zeros = 1 - prob_ones
        return np.column_stack((prob_zeros, prob_ones))

    def predict(self, X):
        """Devuelve clase predicha (1 si probabilidad > 0.5)."""
        return (self.predict_proba(X)[:, 1] >= 0.5).astype(int)


def create_dummy_model(prob_class1=0.15):
    """Factoría para crear una instancia del modelo dummy."""
    return DummyModel(prob_class1=prob_class1)

# COMMAND ----------

logger = MLOpsLogger(
    name='01_load_model',
    log_level='DEBUG' if DEBUG_MLOPS else 'INFO',
    log_dir=LOGS_PATH,
    is_job_run=True,
    job_context={
        'mes_vta': MES_VTA,
        'environment': ENV,
        'notebook': '01_load_model'
    }
)

# COMMAND ----------

# Configurar MLflow para Unity Catalog
mlflow.set_registry_uri("databricks-uc")
logger.info(f"MLflow tracking URI: {mlflow.get_tracking_uri()}")



# COMMAND ----------

try:
    logger.log_stage_start('load_model', mes_vta=MES_VTA, environment=ENV, model_path=MODEL_FILE_PATH)
    start_time = time.time()

    logger.info("=" * 60)
    logger.info("CARGANDO MODELO DESDE PICKLE")
    logger.info("=" * 60)

    # ====================================================
    # 1. Cargar modelo real
    # ====================================================
    model = None
    is_dummy = False          # Inicializar como False (real)
    feature_names = None

    if os.path.exists(MODEL_FILE_PATH):
        file_size = os.path.getsize(MODEL_FILE_PATH)
        if file_size == 0:
            logger.warning(f"Archivo {MODEL_FILE_PATH} vacío. Se eliminará.")
            os.remove(MODEL_FILE_PATH)
        else:
            try:
                with open(MODEL_FILE_PATH, "rb") as file:
                    model = pickle.load(file)
                logger.info(f"✅ Modelo real cargado desde: {MODEL_FILE_PATH}")
                logger.info(f"Tipo: {type(model).__name__}")

                # Verificar si es una instancia del dummy
                is_dummy = isinstance(model, DummyModel)
                logger.info(f"¿Es modelo dummy? {is_dummy}")

                # Intentar extraer nombres de features (si existen)
                if hasattr(model, 'feature_names_') and model.feature_names_ is not None:
                    feature_names = model.feature_names_
                elif hasattr(model, 'feature_names_in_') and model.feature_names_in_ is not None:
                    feature_names = list(model.feature_names_in_)
                elif hasattr(model, '_feature_names') and model._feature_names is not None:
                    feature_names = model._feature_names
                else:
                    # Intento adicional para CatBoost (método get_feature_names)
                    try:
                        if hasattr(model, 'get_feature_names'):
                            feature_names = model.get_feature_names()
                    except:
                        pass

                if feature_names:
                    logger.info(f"Features extraídos del modelo: {len(feature_names)} (primeros 5: {feature_names[:5]})")
                else:
                    logger.warning("El modelo no expone nombres de features.")

            except (EOFError, pickle.UnpicklingError) as e:
                logger.error(f"Error en pickle: {e}. Archivo corrupto, eliminado.")
                os.remove(MODEL_FILE_PATH)
                model = None
                is_dummy = False
    else:
        logger.warning(f"No se encontró el archivo de modelo en {MODEL_FILE_PATH}")

    # ======================================================
    # 2. Si el modelo real no expone features, usar FEATURE_NAMES desde config
    # ======================================================
    if model is not None and not is_dummy and feature_names is None:
        if 'FEATURE_NAMES' in globals() and FEATURE_NAMES:
            feature_names = FEATURE_NAMES
            logger.info(f"Usando lista de features desde config.FEATURE_NAMES: {len(feature_names)} variables")
        else:
            logger.warning("No se encontró FEATURE_NAMES en config. No se podrá registrar en Unity Catalog.")

    # ====================================================
    # 3. Crear dummy si no hay modelo y está permitido
    # ====================================================
    if model is None and USE_DUMMY_MODEL:
        logger.warning("⚠️ Modelo real no encontrado. Creando modelo DUMMY.")
        model = create_dummy_model()
        is_dummy = True
        feature_names = None  # dummy no tiene features
        os.makedirs(os.path.dirname(MODEL_FILE_PATH), exist_ok=True)
        with open(MODEL_FILE_PATH, "wb") as f:
            pickle.dump(model, f)
        if os.path.getsize(MODEL_FILE_PATH) == 0:
            raise IOError("No se pudo guardar el modelo dummy")
        logger.info(f"Modelo dummy guardado en {MODEL_FILE_PATH}")

    if model is None:
        raise FileNotFoundError(f"No hay modelo en {MODEL_FILE_PATH} y USE_DUMMY_MODEL={USE_DUMMY_MODEL}")

    if not hasattr(model, 'predict_proba'):
        raise AttributeError("El modelo no tiene predict_proba")

    # ===========================================================
    # 4. Registrar en MLflow (solo si NO es dummy, hay nombre registrado y hay features)
    # ===========================================================
    registered_version = None
    run_id = None

    if not is_dummy and REGISTERED_MODEL_NAME and REGISTERED_MODEL_NAME.strip() and feature_names:
        logger.info("Registrando modelo real en Unity Catalog con firma...")
        try:
            import pandas as pd

            input_example = pd.DataFrame([[0.0] * len(feature_names)], columns=feature_names)
            proba_example = model.predict_proba(input_example)[:, 1]
            signature = infer_signature(input_example, proba_example)

            with mlflow.start_run(run_name=f"load_model_{MES_VTA}") as run:
                run_id = run.info.run_id

                # --- 1. Registrar el modelo ---
                mlflow.catboost.log_model(
                    cb_model=model,
                    artifact_path="model",
                    registered_model_name=REGISTERED_MODEL_NAME,
                    signature=signature,
                    input_example=input_example
                )
                logger.info(f"Modelo registrado | Run ID: {run_id} | Nombre: {REGISTERED_MODEL_NAME}")

                # --- 2. Obtener la versión registrada ---
                try:
                    client = MlflowClient()
                    # Usar el alias 'latest-databricks' que Databricks asigna automáticamente
                    latest_version = client.get_model_version_by_alias(
                        REGISTERED_MODEL_NAME,
                        "latest-databricks"
                    )
                    registered_version = latest_version.version
                    logger.info(f"Versión registrada: {registered_version}")
                except Exception as e:
                    # Si falla (ejm. primera versión del modelo), se captura sin que sea error
                    logger.warning(f"No se pudo obtener la versión automáticamente: {e}")
                    logger.info("El modelo se registró correctamente (versión no disponible para lectura)")

        except Exception as e:
            logger.error(f"Error al registrar en MLflow: {e}")
            logger.warning("El modelo no se registró, pero se usará localmente.")
    else:
        if is_dummy:
            logger.info("Modelo dummy - no se registra en MLflow.")
        elif not REGISTERED_MODEL_NAME:
                logger.info("REGISTERED_MODEL_NAME no definido - no se registra.")
        elif not feature_names:
                logger.warning("No hay lista de features - no se puede registrar en Unity Catalog.")

    # =====================================================
    # 5. Guardar lista de features en task values
    # =====================================================
    final_feature_names = feature_names if feature_names else (FEATURE_NAMES if 'FEATURE_NAMES' in globals() else None)
    if final_feature_names:
        dbutils.jobs.taskValues.set(key="features_list", value=str(final_feature_names))
        logger.info(f"Lista de features guardada en task values ({len(final_feature_names)} columnas)")
    else:
        dbutils.jobs.taskValues.set(key="features_list", value="none")
        logger.warning("No se pudo guardar lista de features (se usará lista vacía en inferencia)")

    # =====================================
    # 6. Task values finales
    # =====================================
    duration = time.time() - start_time
    logger.log_stage_end('load_model', duration=duration, model_loaded=True, dummy=is_dummy, run_id=run_id)

    dbutils.jobs.taskValues.set(key="model_loaded", value=True)
    dbutils.jobs.taskValues.set(key="dummy_model", value=is_dummy)
    dbutils.jobs.taskValues.set(key="model_version", value=registered_version if registered_version else "none")
    if run_id:
        dbutils.jobs.taskValues.set(key="mlflow_run_id", value=run_id)

    logger.info(f"✅ Modelo cargado exitosamente. Tiempo: {duration:.2f}s")

except Exception as e:
    logger.log_exception(operation='load_model', exception=e, should_raise=True, mes_vta=MES_VTA, environment=ENV)
    dbutils.jobs.taskValues.set(key="model_loaded", value=False)

finally:
    if 'logger' in locals():
        logger.info(f"Finalizando notebook: {logger.name}")
        logger._flush_all_handlers()
        logger.close()

##Check_availability

Propósito: Verificar que las tablas fuente necesarias para construir las variables están disponibles y contienen datos para el mes de proceso.

In [0]:
# Databricks notebook source
# ============================================================================
# 02_check_availability.ipynb
# Propósito: Verificar disponibilidad de tablas fuente y modelo registrado.
#            Maneja errores de configuración de MLflow sin interrumpir.
# ============================================================================

# COMMAND ----------

%run ../config/config

# COMMAND ----------

import sys
import time
import os
import pandas as pd
from pyspark.sql import SparkSession
from pyspark.sql.utils import AnalysisException

import mlflow
import mlflow.tracking._model_registry.utils
# Esta línea fuerza a MLflow a usar el Registry del Unity Catalog.
mlflow.tracking._model_registry.utils._get_registry_uri_from_spark_session = lambda: "databricks-uc"
sys.path.insert(0, str(project_path))
from utils.logger import MLOpsLogger
from utils.control_available_info import control_available_info

# COMMAND ----------

logger = MLOpsLogger(
    name='02_check_availability',
    log_level='DEBUG' if DEBUG_MLOPS else 'INFO',
    log_dir=logs_path,
    is_job_run=True,
    job_context={
        'mes_vta': MES_VTA,
        'environment': ENV,
        'notebook': '02_check_availability'
    }
)

# COMMAND ----------

try:
    logger.log_stage_start('check_availability', mes_vta=MES_VTA, environment=ENV)
    start_time = time.time()
    spark = SparkSession.builder.getOrCreate()

    num_errors = 0

    # ================================================================
    # 1. ARCHIVO DE CONTROL (available_info.txt)
    # ================================================================
    logger.info("=" * 60)
    logger.info("VERIFICANDO DISPONIBILIDAD DE TABLAS FUENTE")
    logger.info("=" * 60)

    mapping_file = os.path.join(MAPPING_PATH, "available_info.txt")
    if os.path.exists(mapping_file):
        logger.info(f"Usando archivo de control: {mapping_file}")
        reporte_disp, num_errors_ctrl = control_available_info(
            in_file_path=mapping_file,
            in_month_ref=str(MES_VTA)
        )
        logger.info(f"Revisión de archivo de control: {num_errors_ctrl} errores")
        num_errors += num_errors_ctrl
    else:
        logger.info("Archivo available_info.txt no encontrado. Se omitirá validación por archivo.")

    # ================================================================
    # 2. VERIFICAR TABLA FUENTE PRINCIPAL
    # ================================================================
    logger.info("\n" + "=" * 60)
    logger.info("VERIFICANDO TABLA FUENTE PRINCIPAL")
    logger.info("=" * 60)
    logger.info(f"Tabla: {SOURCE_TABLE}")
    logger.info(f"Mes a verificar: {MES_VTA}")

    try:
        count_fuente = spark.sql(f"""
            SELECT COUNT(*) as cnt
            FROM {SOURCE_TABLE}
            WHERE codmes = {MES_VTA}
              AND TRIM(codproductonivel0rbm) IN ('PYME REVOLVENTE', 'PYME NO REVOLVENTE')
              AND codclaveunicocli IS NOT NULL
        """).collect()[0]['cnt']

        if count_fuente == 0:
            logger.warning(f"⚠️ Tabla fuente no tiene datos para codmes={MES_VTA} (con filtro PYME)")
            num_errors += 1
        else:
            logger.info(f"✓ Tabla fuente disponible: {count_fuente:,} registros para mes {MES_VTA}")

    except AnalysisException as e:
        logger.error(f"Tabla fuente no existe o no accesible: {str(e)}")
        num_errors += 1
    except Exception as e:
        logger.error(f"Error inesperado al consultar tabla fuente: {str(e)}")
        num_errors += 1

    # ================================================================
    # 3. VERIFICAR MODELO EN MLFLOW (con manejo de CONFIG_NOT_AVAILABLE)
    # ================================================================
    logger.info("\n" + "=" * 60)
    logger.info("VERIFICANDO MODELO REGISTRADO EN MLFLOW")
    logger.info("=" * 60)

    if REGISTERED_MODEL_NAME and REGISTERED_MODEL_NAME.strip():
        try:
            import mlflow
            from mlflow.tracking import MlflowClient
            client = MlflowClient()
            try:
                latest = client.get_model_version_by_alias(REGISTERED_MODEL_NAME, "latest-databricks")
                logger.info(f"✔ Modelo disponible: {REGISTERED_MODEL_NAME} v{latest.version}")
            except Exception as e_inner:
                if "CONFIG_NOT_AVAILABLE" in str(e_inner) or "modelRegistryUri" in str(e_inner):
                    logger.warning("MLflow Model Registry no configurado en este clúster. Omitiendo verificación.")
                elif "RESOURCE_DOES_NOT_EXIST" in str(e_inner):
                    logger.warning(f"Modelo '{REGISTERED_MODEL_NAME}' registrado pero sin alias 'latest-databricks' (puede ser primera versión)")
                else:
                    logger.warning(f"No se pudo verificar modelo en MLflow: {e_inner}")
        except ImportError:
            logger.info("MLflow no instalado. No se verificará registro.")
        except Exception as e:
            logger.warning(f"Error general al verificar MLflow: {e}")
    else:
        logger.info("REGISTERED_MODEL_NAME no definido. No se verificará MLflow.")

    # ================================================================
    # 4. RESUMEN Y TASK VALUES
    # ================================================================
    duration = time.time() - start_time

    dbutils.jobs.taskValues.set(key="num_errors", value=num_errors)
    dbutils.jobs.taskValues.set(key="count_fuente", value=int(count_fuente) if 'count_fuente' in locals() else 0)

    logger.log_data_quality(
        dataset_name='availability_check',
        total_records=int(count_fuente) if 'count_fuente' in locals() else 0,
        valid_records=int(count_fuente) if 'count_fuente' in locals() and num_errors == 0 else 0,
        invalid_records=num_errors,
        errors_found=num_errors
    )

    logger.log_stage_end('check_availability', duration=duration, num_errors=num_errors,
                         count_fuente=int(count_fuente) if 'count_fuente' in locals() else 0)

    if num_errors > 0:
        raise ValueError(f"Verificación falló con {num_errors} error(es). Revisar logs.")

    logger.info("=" * 60)
    logger.info(f"✓ DISPONIBILIDAD VERIFICADA EXITOSAMENTE en {duration:.2f}s")
    logger.info("=" * 60)

except Exception as e:
    logger.log_exception(operation='check_availability', exception=e, should_raise=True,
                         mes_vta=MES_VTA, environment=ENV)

finally:
    if 'logger' in locals():
        logger.info(f"Finalizando notebook: {logger.name}")
        logger._flush_all_handlers()
        logger.close()

##load_preparation_data

Propósito: Construir la base de datos con todas las variables necesarias (incluyendo joins, lógica de dueño, winsorización, etc.) y persistirla en formato Delta o Parquet para las siguientes etapas. Aquí se concentra la mayor parte de la lógica del Notebook Sherly adaptada a la estructura MLOps.

In [0]:
# Databricks notebook source
# ============================================================================
# 03_load_preparation_data.ipynb
# Propósito: Construir la tabla maestra de features (portafolio + variables)
#            aplicando lógica de negocio: dueños, materialidad, variables AG,
#            antigüedad SUNAT, etc. Escritura particionada por codmes.
# ============================================================================

# COMMAND ----------

%run ../config/config

# COMMAND ----------

# Instalación de dependencias (si alguna librería extra se necesita, pero por ahora solo Spark)
# No se requiere instalación adicional.

# COMMAND ----------

import sys
import time
import numpy as np
from pyspark.sql import SparkSession, functions as F
from pyspark.sql.types import IntegerType, DecimalType, DoubleType, StringType
from pyspark.sql.window import Window
from pyspark.sql.functions import udf, array

sys.path.insert(0, str(project_path))
from utils.logger import MLOpsLogger, log_dataframe_info
from utils.month_delta import month_delta

# COMMAND ----------

# Logger (usando MES_VTA y ENV en mayúsculas)
logger = MLOpsLogger(
    name='03_load_preparation_data',
    log_level='DEBUG' if DEBUG_MLOPS else 'INFO',
    log_dir=logs_path,
    is_job_run=True,
    job_context={
        'mes_vta': MES_VTA,
        'environment': ENV,
        'notebook': '03_load_preparation_data'
    }
)

# COMMAND ----------

# ==================================================
# Funciones auxiliares (desde Notebook Sherly)
# ==================================================
def add_codmes_spark(col_name, months):
    """Suma meses a un código de mes (formato YYYYMM) usando expr."""
    return F.expr(f"""
        cast(
            add_months(
                to_date(cast({col_name} as string), 'yyyyMM'),
                {months}
            ) as string
        )
    """)

def operacionesMaxBetweenCols(columnas):
    """UDF para calcular el máximo entre columnas manejando nulos."""
    valorInicial = float(-9999999999999999.0)
    mayor = float(0.0)
    for column in columnas:
        if column is not None and str(column) != "" and str(column).upper() != "NULL":
            numero = float(column)
        else:
            numero = valorInicial
        if numero >= mayor:
            mayor = numero
    if mayor == valorInicial:
        return None
    return mayor

operacionesMaxBetweenCols_udf = udf(operacionesMaxBetweenCols, DoubleType())

# COMMAND ----------

# ==================================================
# Parámetros de ejecución (histórico o solo actual)
# ==================================================
RUN_HISTORICAL = FIRST_RUN
if RUN_HISTORICAL and MESES_HISTORICOS_CALIDAD > 0:
    meses_historicos_lista = []
    mes_tmp = MES_VTA
    for i in range(MESES_HISTORICOS_CALIDAD):
        mes_tmp = month_delta(str(mes_tmp), -1)
        meses_historicos_lista.append(mes_tmp)
    meses_historicos_lista.reverse()
    meses_a_procesar = meses_historicos_lista + [MES_VTA]
    logger.info(f"MODO HISTÓRICO: {len(meses_a_procesar)} meses: {meses_a_procesar}")
else:
    meses_a_procesar = [MES_VTA]
    logger.info(f"Procesando únicamente el mes actual: {MES_VTA}")

spark = SparkSession.builder.getOrCreate()
resultados = []

# Nombre de la tabla de salida (siguiendo el estándar del proyecto)
OUTPUT_FEATURES_TABLE = f"{CATALOG_NAME}.{SCHEMA_NAME}.{BASE_NAME_TABLE_MASTER}_{ENV}"
logger.info(f"Tabla de salida: {OUTPUT_FEATURES_TABLE}")

# COMMAND ----------

for idx_mes, mes_actual_proceso in enumerate(meses_a_procesar, 1):
    logger.info("")
    logger.info("=" * 80)
    logger.info(f"PROCESANDO MES {idx_mes}/{len(meses_a_procesar)}: {mes_actual_proceso}")
    logger.info("=" * 80)

    try:
        logger.log_stage_start('load_preparation_data', mes_vta=mes_actual_proceso, environment=ENV)
        start_time = time.time()

        # ================================================================
        # 1. UNIVERSO DE CLIENTES PYME
        # ================================================================
        logger.info("PASO 1: Universo de clientes PYME")
        sp_clientes = spark.table(SOURCE_TABLE).select(
            "codmes", "codclaveunicocli", "codinternocomputacional", "codclavepartycli"
        ).filter(
            F.trim(F.col("codproductonivel0rbm")).isin('PYME REVOLVENTE', 'PYME NO REVOLVENTE')
            & (F.col("codmes") == mes_actual_proceso)
            & (F.col("codclaveunicocli").isNotNull())
            & (F.col("ctdmesmaduracion") > 0)
        ).distinct()

        universo = sp_clientes.groupBy("codclaveunicocli", "codmes").agg(
            F.max("codinternocomputacional").alias("codinternocomputacional"),
            F.max("codclavepartycli").alias("codclavepartycli")
        )
        universe_count = universo.count()
        logger.info(f"✅ Universo: {universe_count:,} registros")

        # ================================================================
        # 2. VARIABLES DE DUEÑO Y RELACIÓN (extraído del Notebook Sherly)
        # ================================================================
        logger.info("PASO 2: Identificación de dueños y relación")
        cliente_prospecto = spark.table(
            "catalog_lhcl_prod_bcp.bcp_ddv_rmbmcaym_modelogestion_vu.hm_clienteprospectopyme"
        ).filter(F.col("codmes") == mes_actual_proceso).select(
            F.col("CODMES").alias("CODMES_0"),
            "CODCLAVEUNICOCLI",
            "tippartyidentificacion"
        )
        cliente_prospecto = cliente_prospecto.withColumn('codmes', add_codmes_spark('CODMES_0', 0))
        cliente_prospecto = cliente_prospecto.drop("CODMES_0")

        relacionados = spark.table(
            "catalog_lhcl_prod_bcp.bcp_ddv_rbmbcapym_apppyme_vu.hm_relacionadoapppyme"
        ).filter(F.col("codmes") == mes_actual_proceso).select(
            F.col("CODMES").alias("CODMES_0"),
            "CODCLAVEUNICOCIL",
            "CODCLAVEUNICOCILREL",
            "DESTIPREL"
        )
        relacionados = relacionados.withColumn('codmes', add_codmes_spark('CODMES_0', 0))
        relacionados = relacionados.drop("CODMES_0")

        relacionados = relacionados.join(
            cliente_prospecto,
            ["codmes", "CODCLAVEUNICOCIL"],
            "left_outer"
        )

        duenios = relacionados.filter(
            F.col("DESTIPREL").isin('DUENIO', 'DUENIO P.N.')
        ).select(
            "codmes", "CODCLAVEUNICOCIL", "CODCLAVEUNICOCILREL",
            F.when(F.col("tippartyidentificacion") == '6', 'J').otherwise('N').alias("FLGTIPPER")
        )

        duenio_unico = duenios.groupBy("codmes", "CODCLAVEUNICOCIL").agg(
            F.max(
                F.when(F.col('FLGTIPPER') == 'N', F.col('CODCLAVEUNICOCILREL'))
                 .otherwise(F.col('CODCLAVEUNICOCILREL'))
            ).alias('CODCLAVEUNICOCILREL')
        )

        relacionados_con_flag = relacionados.alias("A").join(
            duenio_unico.alias("B"),
            (F.col("A.CODCLAVEUNICOCIL") == F.col("B.CODCLAVEUNICOCIL")) &
            (F.col("A.CODCLAVEUNICOCILREL") == F.col("B.CODCLAVEUNICOCILREL")) &
            (F.col("A.codmes") == F.col("B.codmes")),
            "left_outer"
        ).select(
            F.col("A.codmes"), F.col("A.CODCLAVEUNICOCIL"), F.col("A.CODCLAVEUNICOCILREL"),
            F.when(
                F.col("A.DESTIPREL").isin('DUENIO', 'DUENIO P.N.'),
                F.when(F.col("B.CODCLAVEUNICOCIL").isNull(), 0).otherwise(1)
            ).otherwise(0).alias("FLGRELDUENIO")
        )

        relacion_dueno_final = relacionados_con_flag.filter(F.col('FLGRELDUENIO') == 1).select(
            'codmes', 'CODCLAVEUNICOCIL', 'CODCLAVEUNICOCILREL'
        )

        # ================================================================
        # 3. VARIABLES DEL CLIENTE (edad, actividad económica)
        # ================================================================
        logger.info("PASO 3: Variables demográficas del cliente")
        universo_apppyme = spark.table(
            "catalog_lhcl_prod_bcp.bcp_ddv_rbmbcapym_apppyme_vu.hm_clienteaprobacionapppyme"
        ).filter(F.col("codmes") == mes_actual_proceso).select(
            F.col("CODMES").alias("CODMES_0"),
            "CODCLAVEUNICOCIL",
            F.col("NUMEDAD").cast(IntegerType()).alias("EDAD_FIN"),
            F.col("DESECCIONECONOMICAAPPPYME").alias("ACT_ECO_FIN"),
            "TIPPARTYIDENTIFICACION"
        )
        universo_apppyme = universo_apppyme.withColumn('codmes', add_codmes_spark('CODMES_0', 0))
        universo_apppyme = universo_apppyme.drop('CODMES_0')

        # ================================================================
        # 4. VARIABLES DE CARRETERA (RCC, saldos, materialidad)
        # ================================================================
        logger.info("PASO 4: Construcción de variables base (carretera)")
        carretera_rcc = spark.table(
            "catalog_lhcl_prod_bcp.bcp_ddv_matrizvariables_vu.hm_matrizdeudorrccproducto"
        ).filter(F.col("codmes") == mes_actual_proceso).select(
            F.col("CODMES").alias("CODMES_0"),
            "codclaveunicocli",
            F.col("RCC_TIP_COND_MOR_MAX_CRNNR_MAX_U12").alias("ATRASOMAX_CRNENR_12"),
            F.col("RCC_MTO_ADE_ACT_SHIP_RT_UGM").alias("MONTOADE_ACT_MAX6_S_HIP"),
            F.col("RCC_PCT_UTL_3_RT_U3M").alias("UTL_3"),
            F.col("RCC_CTD_SF_CAL_CPP_FRQ_0_U24").alias("SF_NUM_CAL_CPP24")
        )
        carretera_rcc = carretera_rcc.withColumn('codmes', add_codmes_spark('CODMES_0', 0))
        carretera_rcc = carretera_rcc.drop('CODMES_0')

        resumen_saldo = spark.table(
            "catalog_lhcl_prod_bcp.bcp_ddv_matrizvariables_vu.hm_matrizresumensaldoactivopasivo"
        ).filter(F.col("codmes") == mes_actual_proceso).select(
            F.col("CODMES").alias("CODMES_0"),
            "codclaveunicocli",
            F.col("PROD_MTO_SLD_PRM_TSAV_FRO_100_U24").alias("MESES_PMSAVMF_24_100")
        )
        resumen_saldo = resumen_saldo.withColumn('codmes', add_codmes_spark('CODMES_0', 0))
        resumen_saldo = resumen_saldo.drop('CODMES_0')

        materialidad = spark.table(
            "catalog_lhcl_prod_bcp.bcp_ddv_rbmbcapym_apppyme_vu.hm_variablecrcmaterialidadapppyme"
        ).filter(F.col("codmes") == mes_actual_proceso).select(
            F.col("CODMES").alias("CODMES_0"),
            "codclaveunicocli",
            F.col("RCC_TIP_COND_MOR_MAX_CRNNR_100_MAX_U12").alias("ATRASOMAX_CRNENR_12_100"),
            F.col("RCC_TIP_CF_CAL_CPP_FRO_100_U24").alias("SF_NUM_CAL_CPP24_100"),
            F.col("RCC_CTD_MES_ACT_SF_BUEN_MAL_100_UGM").alias("MESES_ACTIVO_SF_BU_MA6_100")
        )
        materialidad = materialidad.withColumn('codmes', add_codmes_spark('CODMES_0', 0))
        materialidad = materialidad.drop('CODMES_0')

        carretera_completa = carretera_rcc.join(resumen_saldo, ["codmes", "codclaveunicocli"], "fullouter")
        carretera_con_mat = carretera_completa.join(materialidad, ["codmes", "codclaveunicocli"], "fullouter")

        carretera_final = carretera_con_mat.select(
            "codmes", "codclaveunicocli",
            F.when(F.col("ATRASOMAX_CRNENR_12_100").isNull(), F.col("ATRASOMAX_CRNENR_12"))
             .otherwise(F.col("ATRASOMAX_CRNENR_12_100")).alias("ATRASOMAX_CRNENR_12"),
            F.when(F.col("SF_NUM_CAL_CPP24_100").isNull(), F.col("SF_NUM_CAL_CPP24"))
             .otherwise(F.col("SF_NUM_CAL_CPP24_100")).alias("SF_NUM_CAL_CPP24"),
            "MONTOADE_ACT_MAX6_S_HIP", "UTL_3", "MESES_PMSAVMF_24_100"
        )

        carretera_meses_sf = spark.table(
            "catalog_lhcl_prod_bcp.bcp_ddv_matrizvariables_vu.hm_matrizdeudorrccproducto"
        ).filter(F.col("codmes") == mes_actual_proceso).select(
            F.col("CODMES").alias("CODMES_0"),
            "codclaveunicocli",
            F.col("RCC_CTD_MES_ACT_SF_BUEN_MAL_0_UGM").alias("MESES_ACTIVO_SF_BU_MA6_0_BASE")
        )
        carretera_meses_sf = carretera_meses_sf.withColumn('codmes', add_codmes_spark('CODMES_0', 0))
        carretera_meses_sf = carretera_meses_sf.drop("CODMES_0")

        carretera_final = carretera_final.join(carretera_meses_sf, ["codmes", "codclaveunicocli"], "left_outer")
        carretera_final = carretera_final.withColumn(
            "MESES_ACTIVO_SF_BU_MA6_0",
            F.when(F.col("MESES_ACTIVO_SF_BU_MA6_100").isNull(), F.col("MESES_ACTIVO_SF_BU_MA6_0_BASE"))
             .otherwise(F.col("MESES_ACTIVO_SF_BU_MA6_100"))
        ).drop("MESES_ACTIVO_SF_BU_MA6_100", "MESES_ACTIVO_SF_BU_MA6_0_BASE")

        # ================================================================
        # 5. VARIABLES DEL DUEÑO (_RL)
        # ================================================================
        logger.info("PASO 5: Variables del dueño (_RL)")
        variables_dueno = relacion_dueno_final.join(
            carretera_final.select(
                F.col("codmes"),
                F.col("codclaveunicocli").alias("CODCLAVEUNICOCILREL"),
                F.col("ATRASOMAX_CRNENR_12").alias("ATRASOMAX_CRNENR_12_RL"),
                F.col("MONTOADE_ACT_MAX6_S_HIP").alias("MONTOADE_ACT_MAX6_S_HIP_RL"),
                F.col("UTL_3").alias("UTL_3_RL"),
                F.col("SF_NUM_CAL_CPP24").alias("SF_NUM_CAL_CPP24_RL"),
                F.col("MESES_ACTIVO_SF_BU_MA6_0").alias("MESES_ACTIVO_SF_BU_MA6_0_RL"),
                F.col("MESES_PMSAVMF_24_100").alias("MESES_PMSAVMF_24_100_RL")
            ),
            ["CODCLAVEUNICOCILREL", "codmes"],
            "left_outer"
        ).select(
            "codmes", "CODCLAVEUNICOCIL",
            "ATRASOMAX_CRNENR_12_RL", "MONTOADE_ACT_MAX6_S_HIP_RL", "UTL_3_RL",
            "SF_NUM_CAL_CPP24_RL", "MESES_ACTIVO_SF_BU_MA6_0_RL", "MESES_PMSAVMF_24_100_RL"
        ).dropDuplicates(["codmes", "CODCLAVEUNICOCIL"])

        # ================================================================
        # 6. VARIABLES AGREGADAS (_AG)
        # ================================================================
        logger.info("PASO 6: Variables agregadas cliente/dueño (_AG)")
        variables_cliente = carretera_final.select(
            "codmes", "codclaveunicocli", "SF_NUM_CAL_CPP24", "MESES_ACTIVO_SF_BU_MA6_0"
        )

        resultado = universo.join(
            universo_apppyme, ["codmes", "CODCLAVEUNICOCIL"], "left_outer"
        ).join(
            variables_dueno, ["codmes", "CODCLAVEUNICOCIL"], "left_outer"
        ).join(
            variables_cliente, ["codmes", "codclaveunicocli"], "left_outer"
        )

        # Calcular campos AG
        resultado = resultado.withColumn(
            "SF_NUM_CAL_CPP24_AG_raw",
            operacionesMaxBetweenCols_udf(array(F.col("SF_NUM_CAL_CPP24"), F.col("SF_NUM_CAL_CPP24_RL")))
        ).withColumn(
            "SF_NUM_CAL_CPP24_AG",
            F.when(F.col("TIPPARTYIDENTIFICACION") != '6', F.col("SF_NUM_CAL_CPP24"))
             .otherwise(F.col("SF_NUM_CAL_CPP24_AG_raw"))
             .cast(IntegerType())
        )

        resultado = resultado.withColumn(
            "MESES_ACTIVO_SF_BU_MA6_0_AG_raw",
            operacionesMaxBetweenCols_udf(array(F.col("MESES_ACTIVO_SF_BU_MA6_0"), F.col("MESES_ACTIVO_SF_BU_MA6_0_RL")))
        ).withColumn(
            "MESES_ACTIVO_SF_BU_MA6_0_AG",
            F.when(F.col("TIPPARTYIDENTIFICACION") != '6', F.col("MESES_ACTIVO_SF_BU_MA6_0"))
             .otherwise(F.col("MESES_ACTIVO_SF_BU_MA6_0_AG_raw"))
             .cast(IntegerType())
        )

        # ================================================================
        # 7. VARIABLES ADICIONALES: ANTIGÜEDAD SUNAT
        # ================================================================
        logger.info("PASO 7: Variables adicionales (antigüedad SUNAT)")
        contribuyente_sunat = spark.table(
            "catalog_lhcl_prod_bcp.bcp_ddv_rbmccapym_modelogestion_vu.ha_contribuyentesunatpyme"
        ).select(
            F.col("CODMES").alias("CODMES_0"),
            F.col("CODCLAVEUNICOCLI"),
            F.col("CTDMESANTIGUEDADEMP").cast(IntegerType()).alias("ctdmesantiguedadempsunat")
        )
        contribuyente_sunat = contribuyente_sunat.withColumn('codmes', add_codmes_spark('CODMES_0', 0))
        contribuyente_sunat = contribuyente_sunat.drop("CODMES_0")
        resultado = resultado.join(contribuyente_sunat, ["codclaveunicocli", "codmes"], "left_outer")

        # ================================================================
        # 8. SELECCIÓN FINAL Y CAST
        # ================================================================
        resultado_final = resultado.select(
            F.col("codmes").alias("codmes"),
            F.col("codclaveunicocli").alias("codclaveunicocli"),
            F.col("codclavepartycli").alias("codclavepartycli"),
            F.col("codinternocomputacional").alias("codinternocomputacional"),
            F.col("EDAD_FIN").cast(IntegerType()).alias("edad_fin"),
            F.col("ACT_ECO_FIN").alias("act_eco_fin"),
            F.col("ATRASOMAX_CRNENR_12_RL").cast(IntegerType()).alias("atrasomax_crnenr_12_rl"),
            F.col("MONTOADE_ACT_MAX6_S_HIP_RL").cast(DecimalType(19, 8)).alias("montoade_act_max6_s_hip_rl"),
            F.col("UTL_3_RL").cast(DecimalType(23, 6)).alias("utl_3_rl"),
            F.col("MESES_PMSAVMF_24_100_RL").cast(IntegerType()).alias("meses_pmsavmf_24_100_rl"),
            F.col("SF_NUM_CAL_CPP24_AG").cast(IntegerType()).alias("sf_num_cal_cpp24_ag"),
            F.col("MESES_ACTIVO_SF_BU_MA6_0_AG").cast(IntegerType()).alias("meses_activo_sf_bu_ma6_0_ag"),
            F.col("ctdmesantiguedadempsunat").cast(IntegerType()).alias("ctdmesantiguedadempsunat")
        ).distinct()

        # ================================================================
        # 9. ESCRITURA EN TABLA DELTA CON PARTICIONADO POR codmes
        # ================================================================
        # Determinar modo de escritura: overwrite para el primer mes, append para los siguientes
        if idx_mes == 1:
            write_mode = "overwrite"
            logger.info(f"Primer mes ({mes_actual_proceso}) - Modo: OVERWRITE")
        else:
            write_mode = "append"
            logger.info(f"Mes {mes_actual_proceso} - Modo: APPEND (agregando partición)")

        resultado_final.write.mode(write_mode) \
            .option("overwriteSchema", "true") \
            .partitionBy("codmes") \
            .saveAsTable(OUTPUT_FEATURES_TABLE)

        count_final = resultado_final.count()
        logger.info(f"Registros escritos para mes {mes_actual_proceso}: {count_final:,}")
        logger.info(f"Tabla actualizada: {OUTPUT_FEATURES_TABLE}")

        total_duration = time.time() - start_time
        logger.log_stage_end('load_preparation_data', duration=total_duration, records_extracted=count_final)
        resultados.append({
            'mes': mes_actual_proceso,
            'registros': count_final,
            'exitoso': True,
            'duracion': total_duration
        })

    except Exception as e:
        logger.log_exception(
            operation='load_preparation_data',
            exception=e,
            should_raise=False,
            mes_vta=mes_actual_proceso
        )
        resultados.append({
            'mes': mes_actual_proceso,
            'registros': 0,
            'exitoso': False,
            'error': str(e)
        })
        continue

# ================================================================
# RESUMEN FINAL
# ================================================================
logger.info("")
logger.info("=" * 80)
logger.info("RESUMEN FINAL")
logger.info("=" * 80)
exitosos = sum(1 for r in resultados if r['exitoso'])
logger.info(f"Meses: {len(resultados)} | OK: {exitosos} | Error: {len(resultados) - exitosos}")
for r in resultados:
    if r['exitoso']:
        logger.info(f" ✅ {r['mes']}: {r['registros']:,} registros en {r['duracion']:.2f}s")
    else:
        logger.error(f" ❌ {r['mes']}: {r.get('error', '?')}")

if 'logger' in locals():
    logger.info(f"Finalizando notebook: {logger.name}")
    logger._flush_all_handlers()
    logger.close()

#Gem

##Load_Model

In [0]:
# MAGIC %run ../config/config

# Imports
import sys
import time
from pyspark.sql import SparkSession
sys.path.insert(0, str(project_path))
from utils.logger import MLOpsLogger

# Logger load model
logger = MLOpsLogger(
    name='01_load_model',
    log_level='DEBUG' if DEBUG_MLOPS else 'INFO',
    log_dir=logs_path,
    is_job_run=True,
    job_context={'mes_vta': mes_vta, 'environment': env, 'notebook': '01_load_model'}
)

try:
    logger.log_stage_start('load_model', mes_vta=mes_vta, environment=env)
    spark = SparkSession.builder.getOrCreate()
    
    # En este caso, el "modelo" son las tablas maestras de referencia
    # Si existe una tabla de modelo física (CatBoost/XGBoost), se carga aquí
    tables_to_check = [
        f"{catalog_name}.{schema_name}.{BASE_NAME_TABLE_MASTER_ENV}"
    ]
    
    for table in tables_to_check:
        logger.info(f"Verificando disponibilidad de tabla: {table}")
        spark.sql(f"SELECT 1 FROM {table} LIMIT 1")
        logger.info(f"✅ Tabla {table} accesible.")

    dbutils.jobs.taskValues.set(key="model_table_validated", value=True)
    logger.log_stage_end('load_model', status='SUCCESS')

except Exception as e:
    logger.log_exception(operation='load_model', exception=e, should_raise=True)
finally:
    if 'logger' in locals():
        logger._flush_all_handlers()
        logger.close()

Este notebook actúa como el validador de la infraestructura y el catálogo. Su función principal es asegurar que el entorno esté listo antes de mover un solo dato.

Verificación de Tablas Maestras: Comprueba que la tabla maestra definida en la configuración (BASE_NAME_TABLE_MASTER_ENV) exista y sea accesible para escritura.  

Gestión de Dependencias: Carga las configuraciones globales (config) y las librerías de registro de eventos (MLOpsLogger).  

Control de Flujo: Emite un valor de tarea (model_table_validated) que sirve como "semáforo" para que el orquestador de Databricks permita la ejecución de los siguientes pasos.

##check_availability

In [0]:
# MAGIC %run ../config/config

# Imports
import sys
import time
import os
import pandas as pd
from pyspark.sql import SparkSession
sys.path.insert(0, str(project_path))
from utils.logger import MLOpsLogger
from utils.control_available_info import control_available_info

logger = MLOpsLogger(
    name='02_check_availability',
    log_level='DEBUG' if DEBUG_MLOPS else 'INFO',
    log_dir=logs_path,
    is_job_run=True,
    job_context={'mes_vta': mes_vta, 'environment': env, 'notebook': '02_check_availability'}
)

try:
    logger.log_stage_start('check_availability', mes_vta=mes_vta, environment=env)
    spark = SparkSession.builder.getOrCreate()
    
    # Fuentes críticas identificadas en Notebook Sherly
    sources = [
        "catalog_lhcl_prod_bcp.bcp_ddv_adrmmgr_seginfobasesgenerales_vu.hm_portafoliocredito",
        "catalog_lhcl_prod_bcp.bcp_ddv_matrizvariables_vu.hm_matrizdeudorrccproducto",
        "catalog_lhcl_prod_bcp.bcp_ddv_rbmbcapym_apppyme_vu.hm_variablecrcmaterialidadapppyme",
        "catalog_lhcl_prod_bcp.bcp_ddv_rbmccapym_modelogestion_vu.ha_contribuyentesunatpyme"
    ]
    
    for src in sources:
        # Verificamos si hay datos para el mes de proceso o el mes anterior (desfase)
        count = spark.table(src).filter(F.col("codmes") >= (int(mes_vta) - 1)).count()
        if count == 0:
            raise ValueError(f"Fuente {src} no tiene datos para el periodo de referencia.")
        logger.info(f"✅ Fuente {src} validada con {count} registros recientes.")

    dbutils.jobs.taskValues.set(key="availability_status", value=True)

except Exception as e:
    logger.log_exception(operation='check_availability', exception=e, should_raise=True)
finally:
    if 'logger' in locals():
        logger.close()

Este componente es el filtro de calidad y disponibilidad de datos. Se asegura de que la información necesaria para el "recableo" esté actualizada.

Validación de Fuentes Externas: Revisa tablas críticas como el portafolio de créditos, la matriz de variables (carreteras), materialidad y datos de SUNAT.  

Control de Temporalidad: Verifica que las fuentes tengan registros para el periodo de ejecución (mes_vta) o, en su defecto, para el mes anterior, evitando procesar datos vacíos que podrían generar scores erróneos.  

Alertamiento Temprano: Si una fuente esencial no tiene datos, el notebook detiene el proceso (raise Exception) e informa qué tabla específica falló, facilitando el mantenimiento.

##load_preparation_data

In [0]:
# MAGIC %run ../config/config

# Imports
import sys
import time
from pyspark.sql import SparkSession, functions as F
from pyspark.sql.types import IntegerType, DecimalType, DoubleType
sys.path.insert(0, str(project_path))
from utils.logger import MLOpsLogger

logger = MLOpsLogger(
    name='03_load_preparation_data',
    log_level='DEBUG' if DEBUG_MLOPS else 'INFO',
    log_dir=logs_path,
    is_job_run=True,
    job_context={'mes_vta': mes_vta, 'environment': env, 'notebook': '03_load_preparation_data'}
)

def get_desfase_mes(mes):
    # Lógica simplificada para restar 1 mes al codmes (YYYYMM)
    from datetime import datetime
    from dateutil.relativedelta import relativedelta
    dt = datetime.strptime(str(mes), "%Y%m")
    return (dt - relativedelta(months=1)).strftime("%Y%m")

try:
    start_time = time.time()
    spark = SparkSession.builder.getOrCreate()
    mes_data = get_desfase_mes(mes_vta)
    
    logger.info(f"Ejecutando preparación para mes_vta: {mes_vta} usando data de: {mes_data}")

    # 1. UNIVERSO BASE (Pyme Revolvente/No Revolvente)
    df_universo = spark.table("catalog_lhcl_prod_bcp.bcp_ddv_adrmmgr_seginfobasesgenerales_vu.hm_portafoliocredito") \
        .filter((F.col("codmes") == mes_vta) & 
                F.trim(F.col("codproductonivel0rbm")).isin('PYME REVOLVENTE','PYME NO REVOLVENTE')) \
        .select("codmes", "codclaveunicocli", "codinternocomputacional", "codclavepartycli").distinct()

    # 2. LÓGICA DE DUEÑOS Y RELACIONADOS (Paso 2-6 Sherly)
    # Se extrae de hm_relacionadoapppyme con mes_data
    df_rel = spark.table("catalog_lhcl_prod_bcp_bdv_rbmbcapym_apppyme_vu.hm_relacionadoapppyme") \
        .filter(F.col("codmes") == mes_data) \
        .filter(F.col("DESTIPREL").isin('DUENIO', 'DUENIO P.N.'))
    
    # 3. VARIABLES DE CARRETERA Y MATERIALIDAD (Paso 8-11 Sherly)
    carretera = spark.table("catalog_lhcl_prod_bcp.bcp_ddv_matrizvariables_vu.hm_matrizdeudorrccproducto") \
        .filter(F.col("codmes") == mes_data)
    
    materialidad = spark.table("catalog_lhcl_prod_bcp.bcp_ddv_rbmbcapym_apppyme_vu.hm_variablecrcmaterialidadapppyme") \
        .filter(F.col("codmes") == mes_data)

    # Unión y Aplicación de Materialidad
    df_carretera_fin = carretera.join(materialidad, ["codclaveunicocli"], "left") \
        .select(
            F.col("codclaveunicocli"),
            F.coalesce(F.col("RCC_TIP_COND_MOR_MAX_CRNNR_100_MAX_U12"), F.col("RCC_TIP_COND_MOR_MAX_CRNNR_MAX_U12")).alias("ATRASOMAX_RL"),
            F.col("RCC_MTO_ADE_ACT_SHIP_RT_UGM").alias("MONTOADE_RL")
        )

    # 4. ENSAMBLE FINAL (Paso 15-16 Sherly)
    # Aquí se unirían todas las variables procesadas al universo base
    df_final = df_universo.join(df_carretera_fin, "codclaveunicocli", "left") \
        .withColumn("codmes_proceso", F.lit(mes_vta))

    # Persistencia en zona temporal/troncal para inferencia
    output_table = f"{catalog_name}.{schema_name}.{BASE_NAME_TABLE_MASTER_ENV}"
    df_final.write.format("delta").mode("overwrite").saveAsTable(output_table)
    
    logger.info(f"Carga finalizada en {output_table}. Registros: {df_final.count()}")
    logger.log_stage_end('load_preparation_data', duration=time.time()-start_time)

except Exception as e:
    logger.log_exception(operation='load_preparation_data', exception=e, should_raise=True)
finally:
    if 'logger' in locals():
        logger.close()

Es el motor de transformación de datos (ETL/Feature Engineering). Aquí se concentra la lógica de negocio que anteriormente residía en el "Notebook Sherly".Cálculo de Desfase: Implementa la lógica para restar un mes al periodo de venta, asegurando que el modelo use la foto de la carretera y materialidad correcta (mes $T-1$).  Extracción de Lógica RL/AG: Ejecuta el recableo de variables para "Dueños" (RL) y "Relacionados", filtrando por tipos de relación específicos como 'DUENIO P.N.'.  Consolidación de Variables: Une la información del universo base (Pyme Revolvente y No Revolvente) con los indicadores de atraso máximo y montos de adecuación.  Persistencia: Guarda el resultado final en una tabla Delta, la cual servirá como entrada directa para el siguiente paso del pipeline: la inferencia (scoring).  

#MS

##Load_Model

In [0]:
# Databricks notebook source
# MAGIC %md
# MAGIC # 01_load_model
# MAGIC Carga del modelo desde MLflow

# COMMAND ----------

# =========================
# 1. Parámetros
# =========================
dbutils.widgets.text("model_name", "")
dbutils.widgets.text("model_stage", "Production")

model_name = dbutils.widgets.get("model_name")
model_stage = dbutils.widgets.get("model_stage")

# COMMAND ----------

# =========================
# 2. Imports
# =========================
import mlflow
import mlflow.pyfunc

# COMMAND ----------

# =========================
# 3. Configuración MLflow
# =========================
mlflow.set_registry_uri("databricks")

model_uri = f"models:/{model_name}/{model_stage}"

# COMMAND ----------

# =========================
# 4. Carga del modelo
# =========================
try:
    model = mlflow.pyfunc.load_model(model_uri)
    print(f"Modelo cargado correctamente desde: {model_uri}")
    
    # Persistimos referencia en contexto (opcional)
    dbutils.jobs.taskValues.set(key="model_uri", value=model_uri)

except Exception as e:
    raise Exception(f"Error cargando el modelo: {str(e)}")

# COMMAND ----------

# =========================
# 5. Output
# =========================
dbutils.notebook.exit(model_uri)

##Check_availability

In [0]:
# Databricks notebook source
# MAGIC %md
# MAGIC # 02_check_availability
# MAGIC Validación de disponibilidad de recursos

# COMMAND ----------

# =========================
# 1. Parámetros
# =========================
dbutils.widgets.text("model_uri", "")
dbutils.widgets.text("input_path", "")

model_uri = dbutils.widgets.get("model_uri")
input_path = dbutils.widgets.get("input_path")

# COMMAND ----------

# =========================
# 2. Imports
# =========================
import mlflow
import pyspark.sql.functions as F

# COMMAND ----------

# =========================
# 3. Validación de modelo
# =========================
def check_model(uri):
    try:
        mlflow.pyfunc.load_model(uri)
        return True
    except Exception as e:
        print(f"Modelo no disponible: {str(e)}")
        return False

# COMMAND ----------

# =========================
# 4. Validación de datos
# =========================
def check_data(path):
    try:
        df = spark.read.format("delta").load(path)
        if df.limit(1).count() > 0:
            return True
        else:
            print("Dataset vacío")
            return False
    except Exception as e:
        print(f"Error leyendo datos: {str(e)}")
        return False

# COMMAND ----------

# =========================
# 5. Ejecución de checks
# =========================
model_ok = check_model(model_uri)
data_ok = check_data(input_path)

# COMMAND ----------

# =========================
# 6. Resultado final
# =========================
if model_ok and data_ok:
    status = "OK"
else:
    status = "ERROR"

print(f"Estado disponibilidad: {status}")

# COMMAND ----------

# =========================
# 7. Output
# =========================
dbutils.notebook.exit(status)